In [6]:
import sys
sys.path.append("..")
import pandas as pd
import glob
from src.db import append_note

In [7]:
# Các cột ĐÃ BIẾT là thưa dữ liệu một cách có chủ đích (đã ghi trong project_notes.md)
KNOWN_SPARSE_COLUMNS = {
    "office_battery_result_minutes": "Chỉ 1/135 dòng — đã pivot sang battery_density (Giai đoạn 3)",
    "office_battery_per_whr": "Phụ thuộc office_battery_result_minutes — cùng lý do trên",
    "cpu_battery_ratio": "Chỉ 3/157 dòng đủ Geekbench — H1 gốc đã pivot thành H1' (Giai đoạn 2)",
    "cpu_tdp_parsed": "88.68% null trên toàn bộ laptop_model (Giai đoạn 5 Data Quality Monitor)",
    "gpu_tdp_parsed": "78.62% null trên toàn bộ laptop_model (Giai đoạn 5 Data Quality Monitor)",
    "is_workstation": "NULL thay vì False khi không set — xác nhận hợp lý với dataset toàn gaming",
}

files = sorted(glob.glob("../data/processed/*.csv"))
print(f"Kiểm tra {len(files)} file trong data/processed/\n")

report_rows = []
for f in files:
    df = pd.read_csv(f)
    for col in df.columns:
        null_pct = df[col].isna().mean()
        report_rows.append({
            "file": f.split("\\")[-1].split("/")[-1],
            "column": col,
            "rows": len(df),
            "null_pct": round(null_pct, 4),
            "is_known_sparse": col in KNOWN_SPARSE_COLUMNS,
        })

df_report = pd.DataFrame(report_rows)
df_report = df_report[df_report["rows"] > 0]

# Cảnh báo: cột null cao nhưng CHƯA từng được ghi nhận là "biết trước"
unexpected_sparse = df_report[(df_report["null_pct"] > 0.5) & (~df_report["is_known_sparse"])]
print("=== CẢNH BÁO: cột null > 50% nhưng CHƯA được ghi nhận trong assumption log ===")
display(unexpected_sparse)


Kiểm tra 13 file trong data/processed/

=== CẢNH BÁO: cột null > 50% nhưng CHƯA được ghi nhận trong assumption log ===


,file,column,rows,null_pct,is_known_sparse


In [4]:
print("\n=== Toàn bộ báo cáo (tham khảo) ===")
display(df_report.sort_values("null_pct", ascending=False))

df_report.to_csv("../data/processed/_validation_report.csv", index=False)


=== Toàn bộ báo cáo (tham khảo) ===


,file,column,rows,null_pct,is_known_sparse
111,h9_high_interest_low_battery.csv,office_battery_result_minutes,5,1.0000,True
88,golden_ratio_laptops.csv,gpu_tdp_parsed,24,1.0000,True
112,h9_high_interest_low_battery.csv,office_battery_per_whr,5,1.0000,True
113,h9_high_interest_low_battery.csv,cpu_battery_ratio,5,1.0000,True
131,laptop_metrics.csv,office_battery_result_minutes,135,0.9926,True
...,...,...,...,...,...
47,anomaly_battery_minutes.csv,geekbench_7_compute_gpu_battery,0,NaN,False
48,anomaly_battery_minutes.csv,elton_created_at,0,NaN,False
49,anomaly_battery_minutes.csv,anomaly_type,0,NaN,False
50,anomaly_battery_minutes.csv,anomaly_value,0,NaN,False


In [ ]:
### [Giai đoạn 5 - Validation] Xác nhận is_workstation null là hợp lý, không phải lỗi
df_h9 = pd.read_csv("../data/processed/h9_high_interest_low_battery.csv")
display(df_h9[["name", "is_gaming_laptop", "is_workstation"]])

note = """
## [Giai đoạn 5 - Validation] Xác nhận is_workstation null trong h9_high_interest_low_battery

4/5 dòng có is_workstation = NULL (không phải False tường minh). Đã kiểm tra: toàn bộ 5 laptop
trong file đều là gaming (is_gaming_laptop = True) -> is_workstation NULL ở đây phản ánh cách
ClickHouse lưu boolean (NULL khi không set giá trị, không tự động về False), không phải lỗi
merge hay lỗi pipeline. Không ảnh hưởng đến bất kỳ phân tích nào đã thực hiện (H9' không dùng
is_workstation). Bổ sung vào danh sách cột đã biết lý do thưa dữ liệu.
"""
append_note(note)